# Notebook 04: Gold atomic Finance fact

## Runtime contract

- **Input grain:** current Silver ACDOCA document lines, optionally enriched from BKPF headers.
- **Output grain:** one row per deterministic ACDOCA business key in `fct.finance_transaction`.
- **Dimension resolution:** point-in-time SCD2 joins use the posting date and preserve unresolved keys as null.
- **Replay behavior:** an idempotent Delta MERGE updates or inserts by `FINANCE_TRANSACTION_SK`.
- **Failure evidence:** source/output grain and duplicate-key counts append to `audit.finance_fact_reconciliation`.

In [ ]:
from pyspark.sql import functions as F, types as T, Window
from delta.tables import DeltaTable
from functools import reduce
import datetime
import uuid

BRZ_SLV_LH = "lh_brz_slv_finance"
GLD_LH = "lh_gld_finance"
SLV = f"{BRZ_SLV_LH}.slv"
AUD = f"{BRZ_SLV_LH}.audit"
DIM = f"{GLD_LH}.dim"
FCT = f"{GLD_LH}.fct"
HIGH_DATE = "9999-12-31"
BATCH_ID = str(uuid.uuid4())
RUN_TIMESTAMP = datetime.datetime.now(datetime.timezone.utc).replace(tzinfo=None)

In [ ]:
# Silver stores one current row per business key after full-snapshot MERGE.
ACDOCA_KEYS = ["RCLNT", "RLDNR", "RBUKRS", "GJAHR", "BELNR", "DOCLN"]
BKPF_KEYS = ["MANDT", "BUKRS", "GJAHR", "BELNR"]


def dim_join(df, dim_name, alias, conditions, date_col="POSTING_DATE"):
    dimension = spark.table(f"{DIM}.{dim_name}").alias(alias)
    natural_key_match = reduce(lambda left, right: left & right, conditions)
    effective_date_match = (
        (F.col(date_col) >= F.col(f"{alias}.EFFECTIVE_FROM"))
        & (F.col(date_col) <= F.col(f"{alias}.EFFECTIVE_TO"))
    )
    return df.join(dimension, natural_key_match & effective_date_match, "left")


ac = spark.table(f"{SLV}.acdoca").alias("a")
bk = spark.table(f"{SLV}.bkpf").alias("h")

# Header enrichment stays optional and must not alter the ACDOCA line grain.
header_match = (
    F.col("a.RCLNT").eqNullSafe(F.col("h.MANDT"))
    & F.col("a.RBUKRS").eqNullSafe(F.col("h.BUKRS"))
    & F.col("a.GJAHR").eqNullSafe(F.col("h.GJAHR"))
    & F.col("a.BELNR").eqNullSafe(F.col("h.BELNR"))
)
joined = ac.join(bk, header_match, "left")


def a(name, data_type="string"):
    return (
        F.col(f"a.{name}")
        if name in ac.columns
        else F.lit(None).cast(data_type)
    )


def h(name, data_type="string"):
    return (
        F.col(f"h.{name}")
        if name in bk.columns
        else F.lit(None).cast(data_type)
    )


fallback_posting_date = F.to_date(
    F.concat_ws(
        "-",
        a("GJAHR"),
        F.lpad(a("POPER"), 2, "0"),
        F.lit("01"),
    )
)
fact = joined.withColumn(
    "POSTING_DATE",
    F.coalesce(a("BUDAT", "date"), h("BUDAT", "date"), fallback_posting_date),
)
fact = dim_join(
    fact,
    "company",
    "co",
    [
        a("RCLNT").eqNullSafe(F.col("co.MANDT")),
        a("RBUKRS").eqNullSafe(F.col("co.BUKRS")),
    ],
)
fact = dim_join(
    fact,
    "gl_account",
    "gl",
    [
        a("RCLNT").eqNullSafe(F.col("gl.MANDT")),
        F.col("co.KTOPL").eqNullSafe(F.col("gl.KTOPL")),
        a("RACCT").eqNullSafe(F.col("gl.SAKNR")),
    ],
)
fact = dim_join(
    fact,
    "cost_center",
    "cc",
    [
        a("RCLNT").eqNullSafe(F.col("cc.MANDT")),
        a("KOKRS").eqNullSafe(F.col("cc.KOKRS")),
        a("RCNTR").eqNullSafe(F.col("cc.KOSTL")),
    ],
)
fact = dim_join(
    fact,
    "profit_center",
    "pc",
    [
        a("RCLNT").eqNullSafe(F.col("pc.MANDT")),
        a("KOKRS").eqNullSafe(F.col("pc.KOKRS")),
        a("PRCTR").eqNullSafe(F.col("pc.PRCTR")),
    ],
)
fact = dim_join(
    fact,
    "division",
    "dv",
    [F.col("pc.KHINR").eqNullSafe(F.col("dv.DIVISION_CODE"))],
)
fact = dim_join(
    fact,
    "vendor",
    "ve",
    [
        a("RCLNT").eqNullSafe(F.col("ve.MANDT")),
        a("LIFNR").eqNullSafe(F.col("ve.LIFNR")),
    ],
)
fact = dim_join(
    fact,
    "customer",
    "cu",
    [
        a("RCLNT").eqNullSafe(F.col("cu.MANDT")),
        a("KUNNR").eqNullSafe(F.col("cu.KUNNR")),
    ],
)

source_columns = [
    "RCLNT", "RLDNR", "RBUKRS", "GJAHR", "POPER", "BELNR", "DOCLN",
    "RACCT", "RCNTR", "PRCTR", "KOKRS", "LIFNR", "KUNNR", "DRCRK",
    "BLART", "AWTYP", "SGTXT", "RHCUR", "RKCUR", "RWCUR",
]
finance_key_values = [
    F.coalesce(a(column_name).cast("string"), F.lit("__NULL__"))
    for column_name in ACDOCA_KEYS
]

out = fact.select(
    F.sha2(F.concat_ws("||", *finance_key_values), 256).alias(
        "FINANCE_TRANSACTION_SK"
    ),
    F.date_format("POSTING_DATE", "yyyyMMdd").cast("int").alias(
        "POSTING_DATE_SK"
    ),
    ((a("GJAHR", "int") * F.lit(100)) + a("POPER", "int"))
    .cast("int")
    .alias("FISCAL_PERIOD_KEY"),
    F.col("co.DIMENSION_SK").alias("COMPANY_SK"),
    F.col("gl.DIMENSION_SK").alias("GL_ACCOUNT_SK"),
    F.col("cc.DIMENSION_SK").alias("COST_CENTER_SK"),
    F.col("pc.DIMENSION_SK").alias("PROFIT_CENTER_SK"),
    F.col("dv.DIMENSION_SK").alias("DIVISION_SK"),
    F.col("ve.DIMENSION_SK").alias("VENDOR_SK"),
    F.col("cu.DIMENSION_SK").alias("CUSTOMER_SK"),
    *[a(column_name).alias(column_name) for column_name in source_columns],
    h("BKTXT").alias("DOCUMENT_HEADER_TEXT"),
    h("STBLG").alias("REVERSAL_DOCUMENT"),
    a("HSL", "decimal(23,2)").alias("AMOUNT_LOCAL"),
    a("KSL", "decimal(23,2)").alias("AMOUNT_GROUP"),
    a("TSL", "decimal(23,2)").alias("AMOUNT_TRANSACTION"),
    a("WSL", "decimal(23,2)").alias("AMOUNT_OTHER"),
    a("MSL", "decimal(23,3)").alias("QUANTITY"),
    a("BATCH_RUN_ID").alias("SOURCE_BATCH_RUN_ID"),
    a("EXTRACT_DATE", "date").alias("EXTRACT_DATE"),
    a("LOAD_DATE", "date").alias("SILVER_LOAD_DATE"),
    a("SOURCE_SYSTEM").alias("SOURCE_SYSTEM"),
    a("SOURCE_FILE_NAME").alias("SOURCE_FILE_NAME"),
    F.lit(BATCH_ID).alias("GOLD_BATCH_ID"),
    F.current_timestamp().alias("GOLD_LOAD_TIMESTAMP"),
)

# Gold curation is replay-safe at the deterministic transaction key.
target = f"{FCT}.finance_transaction"
if spark.catalog.tableExists(target):
    (
        DeltaTable.forName(spark, target)
        .alias("tgt")
        .merge(
            out.alias("src"),
            "tgt.FINANCE_TRANSACTION_SK = src.FINANCE_TRANSACTION_SK",
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    out.write.format("delta").mode("overwrite").saveAsTable(target)

# Reconciliation protects the ACDOCA document-line grain after enrichment.
source_count = ac.count()
output_count = out.select("FINANCE_TRANSACTION_SK").distinct().count()
duplicate_count = (
    out.groupBy("FINANCE_TRANSACTION_SK")
    .count()
    .filter(F.col("count") > 1)
    .count()
)
recon = spark.createDataFrame(
    [(BATCH_ID, source_count, output_count, duplicate_count)],
    (
        "gold_batch_id string, source_line_count long, "
        "output_distinct_count long, duplicate_key_count long"
    ),
).withColumn("gold_load_timestamp", F.current_timestamp())
recon.write.mode("append").format("delta").saveAsTable(
    f"{AUD}.finance_fact_reconciliation"
)
display(recon)